#Basic Tasks

In [0]:
%sql
--1
create or replace table cyntexa_dev.sales.students(
    roll_no int,
    name string,
    marks int
)

In [0]:
%sql
insert into cyntexa_dev.sales.students values(1,'Ravi',90),(2,'Rajesh',80),(3,'Ramesh',70),(4,'Rakesh',60),(5,'Raj',50)

In [0]:
%sql
update cyntexa_dev.sales.students
set marks=marks+10
where marks<80

In [0]:
%sql
insert into cyntexa_dev.sales.students values(6,'Ravi',90),(7,'Rajesh',80),(8,'Ramesh',70)

In [0]:
%sql
describe history cyntexa_dev.sales.students

In [0]:
%sql
--2
create table if not exists cyntexa_dev.sales.sales_raw

In [0]:
%sql
copy into cyntexa_dev.sales.sales_raw
from '/Volumes/dev/demo/raw_data/sales/'
fileformat = csv
format_options('header' = 'true')
copy_options('mergeSchema' = 'true')

In [0]:
%sql
select count(*) from cyntexa_dev.sales.sales_raw

shows 50 no. of records inserted and total no of records in batch 1 are 50 as well

In [0]:
%sql
copy into cyntexa_dev.sales.sales_raw
from '/Volumes/dev/demo/raw_data/sales/'
fileformat = csv
format_options('header' = 'true')
copy_options('mergeSchema' = 'true')

In [0]:
%sql
select count(*) from cyntexa_dev.sales.sales_raw

shows 78 records inserted and total records after batch 2 are 125

In [0]:
%sql
select count(*) from cyntexa_dev.sales.students

In [0]:
%sql
--3
select * from cyntexa_dev.sales.students

In [0]:
%sql
select * from cyntexa_dev.sales.students version as of 2

In [0]:
%sql
select * from cyntexa_dev.sales.students timestamp as of '2026-09-10T12:43:13.000+00:00'

#Intermediate Tasks

In [0]:
#4
df = spark.read.option("header", "true").csv('/Volumes/cyntexa_dev/sales/raw/sales1.csv')

In [0]:
#append a new column using mergeSchema
from pyspark.sql.functions import col, lit
df = df.withColumn("product_name", lit('Electronics'))

df.write.mode("overwrite").option("mergeSchema", "true").saveAsTable("cyntexa_dev.sales.sales")

In [0]:
#change an existing column's type using overwriteSchema
df = df.withColumn("order_id", col("order_id").cast("int"))

df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("cyntexa_dev.sales.sales")

- mergeSchema merge the schema if we try to add an new column pyspark not allow it but when we use the option mergeSchema true it will let adding the column
- overwriteSchema change the schema if we try to change the type of an existing column pyspark not allow it but when we use the option overwriteSchema true it will let changing the type of the column

In [0]:
#5
from pyspark.sql.functions import current_timestamp

df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", "/Volumes/dev/demo/raw_data/sales/sales_schema")
    .option("header", "true")
    .option("pathGlobFilter", "*.csv")
    .load("/Volumes/dev/demo/raw_data/sales/")
)

df = df.withColumn("_ingested_at", current_timestamp())

In [0]:
from pyspark.sql.functions import current_timestamp

query = (
    df.writeStream
    .trigger(availableNow=True)
    .option("mergeSchema", "true")
    .option("delta.columnMapping.mode", "name")
    .option("checkpointLocation", "/Volumes/dev/demo/raw_data/sales/sales_checkpoint")
    .outputMode("append")
    .toTable("cyntexa_dev.sales.sales_autoloader")
)

query.recentProgress

In [0]:
%sql
select count(*) from cyntexa_dev.sales.sales_autoloader;

yes they have picked up automatically first the records are 78 then when dropped the new file it becomes 155 and when the another new file is dropped it becomes 233

In [0]:
%sql
--6
desc history cyntexa_dev.sales.sales

In [0]:
%sql
restore cyntexa_dev.sales.sales version as of 0

In [0]:
%sql
desc history cyntexa_dev.sales.sales

restore did not delete the versions that were create after selected version, instead it ceated a new version containning the state of the selected version the previous versions can still be queried using time travel

#Advanced Tasks

**7**

- Batch CTAS : good for one-time loads or occasional manual analysis, not good for anything recurring or time-sensitive
- COPY INTO : good for predictable, scheduled drops (e.g. "vendor sends one file every morning at 6am") — simple, reliable, safe to rerun
- Autoloader : good when files show up at unpredictable times and you need them picked up automatically, without someone manually triggering a job
- Lakeflow Declarative Pipelines : good for more complex pipelines with multiple steps, data quality rules, and when you want Databricks to manage retries/monitoring/orchestration for you, not just simple ingestion

Cyntexa should choose Autoloader where files arrive unpredictably throughout the day, it is the preferred ingestion pattern.It is designed to incrementally detect and process newly arriving files without requiring a complete batch reload.

In [0]:
%sql
--8
select count(*) from cyntexa_dev.sales.sales_bronze

In [0]:
%sql
describe history cyntexa_dev.sales.sales_bronze

In [0]:
%sql
select count(*) from cyntexa_dev.sales.sales_silver version as of 0

In [0]:
%sql
restore table cyntexa_dev.sales.sales_silver to version as of 0

In [0]:
df = spark.read.table('cyntexa_dev.sales.sales_bronze')

df.write.mode('overwrite').saveAsTable('cyntexa_dev.sales.sales_bronze')

In [0]:
%sql
select count(*) from cyntexa_dev.sales.sales_bronze

In [0]:
%sql
--9
describe history cyntexa_dev.sales.sales_silver

In [0]:
%sql
select
  version,
  timestamp,
  operation,
  lag(timestamp) over (order by timestamp) as previous_update,
  timestamp - lag(timestamp) over (order by timestamp) as time_since_last_update
from (
  describe history cyntexa_dev.sales.sales_silver
)
order by timestamp;